# Programmations parallèle & concurrente

## Mesurer avant d'optimiser

La fonction `doublons` ci-dessous renvoie les éléments présents plusieurs fois dans une liste.

1. Avec `timeit.timeit`, mesurez le temps d'exécution de `doublons(nombres)`.
2. Avec `cProfile.run`, trouvez l'appel dans lequel la fonction passe le plus de temps.
3. Proposez une version plus rapide, vérifiez qu'elle renvoie les mêmes éléments et mesurez le gain avec `timeit`.

**Documentation utile**

- [`timeit.timeit`](https://docs.python.org/fr/3/library/timeit.html#timeit.timeit)
- [`cProfile.run`](https://docs.python.org/fr/3.13/library/profile.html#profile.run)
- [`collections.Counter`](https://docs.python.org/fr/3/library/collections.html#collections.Counter)

In [ ]:
import random

nombres = [random.randrange(5_000) for _ in range(5_000)]


def doublons(valeurs: list[int]) -> list[int]:
  resultat = []
  for valeur in valeurs:
    if valeurs.count(valeur) > 1 and valeur not in resultat:
      resultat.append(valeur)
  return resultat


# Votre code ici

### Conseils supplémentaires

`timeit.timeit("doublons(nombres)", globals=globals(), number=3)` renvoie la durée totale des 3 exécutions, en secondes : divisez-la par 3. `globals=globals()` donne au code mesuré accès aux variables du notebook. `cProfile.run("doublons(nombres)", sort="tottime")` affiche un tableau trié par temps passé dans chaque fonction elle-même : regardez la première ligne. Une version rapide compte toutes les occurrences en un seul parcours, avec `collections.Counter`.

```python
duree = timeit.timeit("doublons(nombres)", globals=globals(), number=3) / 3
print(f"doublons : {duree:.3f}s")

cProfile.run("doublons(nombres)", sort="tottime")


def doublons_rapide(valeurs: list[int]) -> list[int]:
  return [valeur for valeur, n in collections.Counter(valeurs).items() if ...]
```

### Solution

In [ ]:
import collections
import cProfile
import timeit

duree = timeit.timeit("doublons(nombres)", globals=globals(), number=3) / 3
print(f"doublons : {duree:.3f}s")

cProfile.run("doublons(nombres)", sort="tottime")


def doublons_rapide(valeurs: list[int]) -> list[int]:
  return [valeur for valeur, n in collections.Counter(valeurs).items() if n > 1]


assert sorted(doublons_rapide(nombres)) == sorted(doublons(nombres))
duree = timeit.timeit("doublons_rapide(nombres)", globals=globals(), number=3) / 3
print(f"doublons_rapide : {duree:.5f}s")

Le profil montre que presque tout le temps est passé dans `list.count`, appelé pour chaque élément : chaque appel reparcourt toute la liste, l'algorithme est donc quadratique. [`collections.Counter`](https://docs.python.org/fr/3/library/collections.html#collections.Counter) compte toutes les occurrences en un seul parcours.

## Calcul : threads ou processus ?

La fonction `compte_premiers(debut, fin)` ci-dessous compte les nombres premiers d'un intervalle : c'est un calcul pur, sans entrée/sortie.

Comptez les nombres premiers de 0 à 400 000, découpés en 4 morceaux (`morceaux`), de trois façons, en mesurant la durée de chacune avec `time.perf_counter` :

1. séquentiellement ;
2. avec un `concurrent.futures.ThreadPoolExecutor` de 4 threads ;
3. avec un `concurrent.futures.ProcessPoolExecutor` de 4 processus : le même code, où seule la classe change.

Les trois résultats doivent être égaux. Comment expliquez-vous les durées obtenues ?

**Documentation utile**

- [`time.perf_counter`](https://docs.python.org/fr/3/library/time.html#time.perf_counter)

In [ ]:
import multiprocessing

# Les processus créés plus bas doivent retrouver les fonctions définies dans ce
# notebook. « fork » les copie avec lui (Linux et macOS) ; « forkserver », le
# défaut de Linux depuis Python 3.14, ne les trouve pas et le pool reste bloqué.
multiprocessing.set_start_method("fork", force=True)


def compte_premiers(debut: int, fin: int) -> int:
  compte = 0
  for n in range(max(debut, 2), fin):
    if all(n % d for d in range(2, int(n**0.5) + 1)):
      compte += 1
  return compte


morceaux = [(i, i + 100_000) for i in range(0, 400_000, 100_000)]

In [ ]:
# Votre code ici

### Conseils supplémentaires

`executeur.map(compte_premiers, debuts, fins)` appelle `compte_premiers(debuts[0], fins[0])`, `compte_premiers(debuts[1], fins[1])`… comme `zip` : séparez d'abord les débuts et les fins des `morceaux`. Le résultat est un itérateur, que `sum` consomme. `max_workers=4` fixe le nombre de threads ou de processus. Mesurez chaque version entre deux appels à `time.perf_counter()`.

```python
debuts = [a for a, _ in morceaux]
fins = [b for _, b in morceaux]

depart = time.perf_counter()
sequentiel = sum(compte_premiers(a, b) for a, b in morceaux)
print(f"Séquentiel : {time.perf_counter() - depart:.2f}s")

depart = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executeur:
  threads = sum(executeur.map(...))
print(f"Threads : {time.perf_counter() - depart:.2f}s")
```

### Solution

In [ ]:
import concurrent.futures
import time

debuts = [a for a, _ in morceaux]
fins = [b for _, b in morceaux]

depart = time.perf_counter()
sequentiel = sum(compte_premiers(a, b) for a, b in morceaux)
print(f"Séquentiel : {time.perf_counter() - depart:.2f}s")

depart = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executeur:
  threads = sum(executeur.map(compte_premiers, debuts, fins))
print(f"Threads : {time.perf_counter() - depart:.2f}s")

depart = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executeur:
  somme_processus = sum(executeur.map(compte_premiers, debuts, fins))
print(f"Processus : {time.perf_counter() - depart:.2f}s")

print(sequentiel == threads == somme_processus)

Les threads ne vont pas plus vite que la version séquentielle : le GIL ne laisse qu'un thread exécuter du code Python à la fois (sauf dans la version *free-threaded* de Python, sans GIL, supportée depuis Python 3.14). Les processus ont chacun leur interpréteur et calculent réellement en parallèle, dans la limite du nombre de cœurs (2 sur Colab). Passer des threads aux processus n'a demandé que de changer la classe de l'exécuteur.

Pour aller plus loin : sur votre machine, relancez ce calcul sous `python3.14t` (la version *free-threaded*) : les threads y calculent en parallèle.

## Fixer un argument : `functools.partial`

`Pool.map` appelle sa fonction avec un seul argument ; la fonction `f` ci-dessous en prend deux.

1. Avec `functools.partial`, appelez `f` en parallèle dans un `multiprocessing.Pool()`, pour les `n` de 0 à 9 et avec `verbeux=True`. Combien de processus différents ont travaillé ? Comparez à `os.cpu_count()`, le nombre de processus que `Pool()` crée par défaut.
2. Essayez maintenant `pool.map(lambda n: f(n, False), range(10))`. Quelle erreur obtenez-vous ? Pourquoi `partial` fonctionne-t-il, et pas la `lambda` ?

**Documentation utile**

- [`functools.partial`](https://docs.python.org/fr/3/library/functools.html#functools.partial)
- [`os.cpu_count`](https://docs.python.org/fr/3/library/os.html#os.cpu_count)

In [ ]:
import os


def f(n: int, verbeux: bool) -> int:
  if verbeux:
    print(f"n = {n}, processus {os.getpid()} (parent {os.getppid()})")
  return n * 2


# Votre code ici

### Conseils supplémentaires

`functools.partial(f, verbeux=True)` renvoie une nouvelle fonction d'un seul argument, qui appelle `f(n, verbeux=True)` : c'est elle que `pool.map` reçoit. Les numéros affichés après « processus » sont les PID des processus du pool. Pour la `lambda`, entourez l'appel d'un `try` / `except Exception as erreur`, et affichez `type(erreur).__name__` et `erreur`.

```python
with multiprocessing.Pool() as pool:
  resultats = pool.map(functools.partial(f, ...), range(10))
print(resultats)
print(os.cpu_count())

with multiprocessing.Pool() as pool:
  try:
    pool.map(lambda n: f(n, False), range(10))
  except Exception as erreur:
    print(...)
```

### Solution

In [ ]:
import functools
import multiprocessing
import os

with multiprocessing.Pool() as pool:
  resultats = pool.map(functools.partial(f, verbeux=True), range(10))
print(resultats)
print(f"Processus du pool par défaut : {os.cpu_count()}")

with multiprocessing.Pool() as pool:
  try:
    pool.map(lambda n: f(n, False), range(10))
  except Exception as erreur:
    print(f"{type(erreur).__name__} : {erreur}")

`Pool.map` envoie chaque tâche à un processus du pool en la sérialisant avec `pickle`. `pickle` enregistre une fonction par son nom, que le processus retrouve dans le module où elle est définie : une `lambda` n'a pas de nom, d'où la `PicklingError`. Un objet `partial`, lui, se sérialise comme la fonction `f` (par son nom) et ses arguments fixés.

`Pool()` crée `os.cpu_count()` processus, mais il leur distribue les tâches par paquets (`chunksize`) : des tâches aussi courtes que celles-ci sont souvent toutes traitées par quelques processus seulement.

## Effectuer deux tâches différentes en parallèle avec un `Pool`

Le code suivant soumet deux calculs différents au même `Pool` avec [`map_async`](https://docs.python.org/fr/3/library/multiprocessing.html#multiprocessing.pool.Pool.map_async), qui rend la main immédiatement au lieu d'attendre les résultats comme `map` : les deux calculs s'exécutent donc en même temps sur les processus du pool.

Réécrivez-le avec [`concurrent.futures.ProcessPoolExecutor`](https://docs.python.org/fr/3/library/concurrent.futures.html#concurrent.futures.ProcessPoolExecutor).

In [ ]:
import multiprocessing


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with multiprocessing.Pool() as pool:
    resultats_a = pool.map_async(a, range(10))
    resultats_b = pool.map_async(b, range(10))
    print(resultats_a.get())
    print(resultats_b.get())

In [ ]:
# Votre code ici

### Conseils supplémentaires

Remplacez `multiprocessing.Pool()` par `concurrent.futures.ProcessPoolExecutor()`. `executeur.submit(a, i)` soumet un calcul sans attendre et renvoie aussitôt un `Future` ; `future.result()` attend ensuite son résultat. Soumettez tous les calculs de `a` et de `b` avant de lire le moindre résultat, pour qu'ils s'exécutent en même temps.

```python
import concurrent.futures

if __name__ == "__main__":
  with concurrent.futures.ProcessPoolExecutor() as executeur:
    futures_a = [executeur.submit(a, i) for i in range(10)]
    futures_b = ...
    print([future.result() for future in futures_a])
    print(...)
```

### Solution

In [ ]:
import concurrent.futures


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with concurrent.futures.ProcessPoolExecutor() as executeur:
    futures_a = [executeur.submit(a, i) for i in range(10)]
    futures_b = [executeur.submit(b, i) for i in range(10)]
    print([future.result() for future in futures_a])
    print([future.result() for future in futures_b])

## Créer une architecture producteur / consommateur

Le code suivant fait communiquer deux processus par une [`multiprocessing.Queue`](https://docs.python.org/fr/3/library/multiprocessing.html#multiprocessing.Queue) : le producteur y dépose des éléments, le consommateur les retire et les affiche.

Exécutez-le, puis répondez : pourquoi le producteur dépose-t-il `None` à la fin ? Modifiez ensuite le code pour lancer deux consommateurs au lieu d'un.

In [ ]:
import multiprocessing


def consommateur(file_attente: multiprocessing.Queue) -> None:
  while True:
    element = file_attente.get()
    if element is None:
      break
    print(element)


def producteur(file_attente: multiprocessing.Queue) -> None:
  for i in range(10):
    file_attente.put(i)
  file_attente.put(None)


if __name__ == "__main__":
  file_attente = multiprocessing.Queue()
  processus_consommateur = multiprocessing.Process(
    target=consommateur, args=(file_attente,)
  )
  processus_producteur = multiprocessing.Process(
    target=producteur, args=(file_attente,)
  )
  processus_consommateur.start()
  processus_producteur.start()
  processus_consommateur.join()
  processus_producteur.join()

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le consommateur ne sort de sa boucle qu'en recevant `None`. Avec deux consommateurs, chacun doit recevoir le sien : le producteur dépose autant de `None` qu'il y a de consommateurs. Donnez un nom à chaque consommateur pour voir lequel affiche quoi.

```python
def producteur(file_attente: multiprocessing.Queue, nb_consommateurs: int) -> None:
  for i in range(10):
    file_attente.put(i)
  for _ in range(...):
    file_attente.put(None)
```

### Solution

`None` est une valeur sentinelle : elle signale au consommateur qu'il n'y aura plus d'éléments. Sans elle, le consommateur resterait bloqué indéfiniment sur `file_attente.get()`. Avec plusieurs consommateurs, chacun doit recevoir sa propre sentinelle :

In [ ]:
import multiprocessing


def consommateur(nom: str, file_attente: multiprocessing.Queue) -> None:
  while True:
    element = file_attente.get()
    if element is None:
      break
    print(f"{nom} : {element}")


def producteur(file_attente: multiprocessing.Queue, nb_consommateurs: int) -> None:
  for i in range(10):
    file_attente.put(i)
  for _ in range(nb_consommateurs):
    file_attente.put(None)


if __name__ == "__main__":
  file_attente = multiprocessing.Queue()
  consommateurs = [
    multiprocessing.Process(
      target=consommateur, args=(f"consommateur {k}", file_attente)
    )
    for k in range(2)
  ]
  processus_producteur = multiprocessing.Process(
    target=producteur, args=(file_attente, len(consommateurs))
  )
  for processus in [*consommateurs, processus_producteur]:
    processus.start()
  for processus in [*consommateurs, processus_producteur]:
    processus.join()

## Un compte partagé entre threads : `threading.Lock`

Quatre threads créditent le même compte, 1 000 fois chacun : ils lisent le solde, attendent un instant (une entrée/sortie, simulée par `time.sleep(0)`), puis écrivent le solde augmenté de 1.

1. Écrivez cette fonction `crediter()`, lancez-la dans 4 `threading.Thread`, attendez-les, puis affichez `solde`. Vaut-il 4 000 ?
2. Corrigez `crediter` avec un `threading.Lock` pris autour de la lecture et de l'écriture (`with verrou:`), et vérifiez le solde.

In [ ]:
import threading
import time

solde = 0

# Votre code ici

### Conseils supplémentaires

`global solde` permet à la fonction de modifier la variable du notebook. Entre la lecture et l'écriture, `time.sleep(0)` laisse la main à un autre thread, qui lit le même solde. Sans cette attente, `solde += 1` seul ne montre presque jamais l'erreur en CPython 3.13, sans être sûr pour autant. Le verrou ne laisse qu'un thread à la fois dans le bloc `with verrou:`. Remettez `solde` à 0 avant la seconde version.

```python
def crediter() -> None:
  global solde
  for _ in range(1_000):
    valeur = solde
    time.sleep(0)
    solde = ...


threads = [threading.Thread(target=crediter) for _ in range(4)]
for thread in threads:
  thread.start()
for thread in threads:
  thread.join()
print(solde)
```

### Solution

In [ ]:
import threading
import time


def crediter() -> None:
  global solde
  for _ in range(1_000):
    valeur = solde
    time.sleep(0)
    solde = valeur + 1


def crediter_avec_verrou() -> None:
  global solde
  for _ in range(1_000):
    with verrou:
      valeur = solde
      time.sleep(0)
      solde = valeur + 1


def lance_threads(cible) -> None:
  threads = [threading.Thread(target=cible) for _ in range(4)]
  for thread in threads:
    thread.start()
  for thread in threads:
    thread.join()


solde = 0
lance_threads(crediter)
print(f"Sans verrou : {solde}")

solde = 0
verrou = threading.Lock()
lance_threads(crediter_avec_verrou)
print(f"Avec verrou : {solde}")

Sans verrou, un thread lit le solde, laisse la main pendant `time.sleep(0)`, et un autre lit la même valeur : chacun écrit ensuite cette valeur plus 1, et l'un des deux crédits est perdu. C'est une condition de course. Avec le verrou, la lecture et l'écriture se font d'un bloc, sans autre thread entre les deux : le solde vaut 4 000. Le verrou rend aussi ces crédits séquentiels : on n'y place que la section critique, aussi courte que possible.

## Télécharger plusieurs pages : séquentiel ou threads ?

Écrivez une fonction `telecharger(url, chemin)` qui télécharge `url` dans le fichier `chemin`, puis téléchargez les pages de `a_telecharger` séquentiellement, puis avec un `concurrent.futures.ThreadPoolExecutor` (méthode `map`), en mesurant la durée de chaque version.

Pour télécharger une page, vous pourrez utiliser le code suivant, où `url` est l'URL à télécharger et `chemin` le chemin du fichier à écrire :

```python
requete = urllib.request.Request(url, headers={"User-Agent": AGENT_UTILISATEUR})
with urllib.request.urlopen(requete) as reponse, chemin.open("wb") as fichier:
  shutil.copyfileobj(reponse, fichier)
```

L'en-tête `User-Agent` est nécessaire : des sites comme Python Tutor ou Wikipédia refusent l'agent par défaut de `urllib` (erreur HTTP 403). `AGENT_UTILISATEUR` dit qui fait la requête et comment le joindre, comme le demandent leurs règles.

Pourquoi les threads accélèrent-ils ces téléchargements, alors qu'ils n'accéléraient pas le calcul des nombres premiers ?

**Documentation utile**

- [`urllib.request.urlopen`](https://docs.python.org/fr/3/library/urllib.request.html#urllib.request.urlopen)
- [`shutil.copyfileobj`](https://docs.python.org/fr/3/library/shutil.html#shutil.copyfileobj)

In [ ]:
import pathlib

AGENT_UTILISATEUR = "shuuchuu-labs/1.0 (https://github.com/shuuchuu/labs)"

telechargements = pathlib.Path("telechargements")
telechargements.mkdir(exist_ok=True)

a_telecharger = (
  ("https://docs.python.org/fr/3/", telechargements / "python-docs.html"),
  ("https://pythontutor.com/", telechargements / "python-tutor.html"),
  ("https://www.google.com/", telechargements / "google.html"),
  ("https://peps.python.org/pep-0008/", telechargements / "pep-8.html"),
  ("https://pypi.org/", telechargements / "pypi.html"),
)

# Votre code ici

### Conseils supplémentaires

`executeur.map(telecharger, urls, chemins)` appelle `telecharger(urls[0], chemins[0])`, `telecharger(urls[1], chemins[1])`… : séparez d'abord `a_telecharger` en deux listes. `map` renvoie un itérateur : `list(...)` attend la fin des téléchargements et fait remonter leurs erreurs.

```python
def telecharger(url: str, chemin: pathlib.Path) -> None: ...


debut = time.perf_counter()
for url, chemin in a_telecharger:
  telecharger(url, chemin)
print(f"Séquentiel : {time.perf_counter() - debut:.2f}s")

urls = [url for url, _ in a_telecharger]
chemins = [chemin for _, chemin in a_telecharger]
debut = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor() as executeur:
  list(executeur.map(...))
print(f"Threads : {time.perf_counter() - debut:.2f}s")
```

### Solution

In [ ]:
import concurrent.futures
import pathlib
import shutil
import time
import urllib.request


def telecharger(url: str, chemin: pathlib.Path) -> None:
  requete = urllib.request.Request(url, headers={"User-Agent": AGENT_UTILISATEUR})
  with urllib.request.urlopen(requete) as reponse, chemin.open("wb") as fichier:
    shutil.copyfileobj(reponse, fichier)


debut = time.perf_counter()
for url, chemin in a_telecharger:
  telecharger(url, chemin)
print(f"Séquentiel : {time.perf_counter() - debut:.2f}s")

urls = [url for url, _ in a_telecharger]
chemins = [chemin for _, chemin in a_telecharger]
debut = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor() as executeur:
  list(executeur.map(telecharger, urls, chemins))
print(f"Threads : {time.perf_counter() - debut:.2f}s")

Un thread qui attend le réseau relâche le GIL : les attentes des téléchargements se chevauchent. Les threads conviennent aux entrées/sorties, les processus au calcul.

## Téléchargements concurrents avec `asyncio`

Reprenez `a_telecharger` et la fonction `telecharger` de l'exercice précédent, et téléchargez les pages de manière concurrente avec `asyncio` : `asyncio.to_thread` exécute une fonction bloquante (comme `telecharger`) dans un thread, sans bloquer la boucle d'événements, et un `asyncio.TaskGroup` lance plusieurs tâches et les attend toutes. Affichez la durée totale des téléchargements.

Attention : dans Colab (comme dans Jupyter), une boucle d'événements tourne déjà, et `asyncio.run(main())` lève donc une `RuntimeError`. Écrivez directement `await main()` dans la cellule ; dans un script, on utiliserait `asyncio.run(main())`.

**Documentation utile**

- [`asyncio.to_thread`](https://docs.python.org/fr/3/library/asyncio-task.html#asyncio.to_thread)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`asyncio.to_thread(telecharger, url, chemin)` renvoie une coroutine ; `groupe.create_task(...)` la lance comme tâche du groupe. Le bloc `async with asyncio.TaskGroup() as groupe:` ne se termine qu'une fois toutes ses tâches finies. Lancez ensuite `await main()`.

```python
async def main() -> None:
  debut = time.perf_counter()
  async with asyncio.TaskGroup() as groupe:
    for url, chemin in a_telecharger:
      groupe.create_task(...)
  print(
    f"{len(a_telecharger)} pages téléchargées en {time.perf_counter() - debut:.2f}s"
  )
```

### Solution

In [ ]:
import asyncio
import time


async def main() -> None:
  debut = time.perf_counter()
  async with asyncio.TaskGroup() as groupe:
    for url, chemin in a_telecharger:
      groupe.create_task(asyncio.to_thread(telecharger, url, chemin))
  print(
    f"{len(a_telecharger)} pages téléchargées en {time.perf_counter() - debut:.2f}s"
  )


await main()

`asyncio.to_thread` exécute toujours `telecharger` dans des threads : la boucle d'événements ne fait qu'attendre leur fin, et le gain vient des threads, comme à l'exercice précédent. `asyncio.gather(*(asyncio.to_thread(telecharger, url, chemin) for url, chemin in a_telecharger))` ferait de même ; `TaskGroup` (Python 3.11) est recommandé pour du nouveau code : si une tâche lève une exception, il annule les autres, et les erreurs remontent ensemble dans un `ExceptionGroup`.

L'intérêt d'`asyncio` est l'entrée/sortie non bloquante, sans thread : une bibliothèque asynchrone comme `httpx` (préinstallée sur Colab) fait les requêtes dans la boucle d'événements elle-même. Pour aller plus loin, essayez cette version, puis `await main_httpx()` :

```python
import httpx


async def telecharger_asynchrone(
  client: httpx.AsyncClient, url: str, chemin: pathlib.Path
) -> None:
  reponse = await client.get(url, follow_redirects=True)
  chemin.write_bytes(reponse.content)


async def main_httpx() -> None:
  async with httpx.AsyncClient(headers={"User-Agent": AGENT_UTILISATEUR}) as client:
    async with asyncio.TaskGroup() as groupe:
      for url, chemin in a_telecharger:
        groupe.create_task(telecharger_asynchrone(client, url, chemin))
```